# Praktikum 1 - Klasifikasi Cat vs Dog

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membuat model CNN sederhana untuk mengklasifikasikan citra kucing dan anjing.

## Langkah 1 - Import Library

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

I0000 00:00:1790791181.888400   90800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790791181.888784   90800 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790791181.923878   90800 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790791183.018121   90800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790791183.018364   90800 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Langkah 2 - Pra Pengolahan Data

Pada tahap ini kita melakukan manipulasi pada citra: normalisasi nilai piksel, koreksi kemiringan (*shear*), pembesaran (*zoom*), dan *flip*.

> **Catatan:** Dataset sudah tersedia secara lokal pada `../data/cat_dog/dataset/`, sehingga tidak perlu mengunduh dari Google Drive.

### Langkah 2.1. Pra Pengolahan Data Training

In [2]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('../data/cat_dog/dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


### Langkah 2.2. Pra Pengolahan Data Testing

In [3]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('../data/cat_dog/dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


## Langkah 3 - Pembuatan Model CNN

### Langkah 3.1. - Inisiasi Model CNN

In [4]:
cnn = tf.keras.models.Sequential()

### Langkah 3.2. - Pembuatan Layer Konvolusi 1

In [5]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

E0000 00:00:1790791184.479302   90800 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790791184.479730   90890 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790791184.498084   90800 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


### Langkah 3.3 - Pembuatan Layer Pooling 1

In [6]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2

In [7]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Langkah 3.5 - Flattening

In [8]:
cnn.add(tf.keras.layers.Flatten())

### Langkah 3.6 - Fully Connected Layer 1 (Input)

In [9]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Langkah 3.7 - Fully Connected Layer 2 (Output)

In [10]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### Langkah 3.8 - Compile Model CNN

In [11]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

Penggunaan *loss function binary crossentropy* dikarenakan kita hanya melakukan klasifikasi pada dua kelas, yaitu kucing dan anjing.

## Langkah 4 - Fit CNN

> **Catatan penyesuaian (CPU):** Pada modul asli, pelatihan dijalankan dengan `epochs=25` pada seluruh 8000 citra training. Agar dapat selesai dalam beberapa menit di lingkungan CPU (tanpa GPU), di sini digunakan `epochs=3` dengan `steps_per_epoch=30` dan `validation_steps=30`. Hasil akurasi karenanya lebih rendah daripada modul asli, tetapi seluruh alur pelatihan dan prediksi tetap sama.

In [12]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 3,
        steps_per_epoch = 30, validation_steps = 30)

Epoch 1/3


I0000 00:00:1790791184.844059   90800 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


 1/30 ━━━━━━━━━━━━━━━━━━━━ 21s 730ms/step - accuracy: 0.3438 - loss: 0.7233

 3/30 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.3750 - loss: 0.8381  

 4/30 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.4453 - loss: 0.7999

 5/30 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.4688 - loss: 0.7770

... (81 outputs left out)

28/30 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5312 - loss: 0.6955

29/30 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5312 - loss: 0.6954

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5281 - loss: 0.6954

30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.5281 - loss: 0.6954 - val_accuracy: 0.5063 - val_loss: 0.6911


## Langkah 5 - Prediksi dengan 1 Citra

Pada langkah ini, kita akan mencoba melakukan prediksi pada 1 citra anjing dan kucing.

In [13]:
import numpy as np
from tensorflow.keras.preprocessing import image

print(training_set.class_indices)

for nama_file in ['cat_or_dog_1.jpg', 'cat_or_dog_2.jpg']:
    test_image = image.load_img('../data/cat_dog/dataset/single_prediction/' + nama_file,
                                target_size = (64, 64))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    result = cnn.predict(test_image)
    prediction = 'dog' if result[0][0] > 0.5 else 'cat'
    print(nama_file, '->', prediction)

{'cats': 0, 'dogs': 1}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


cat_or_dog_1.jpg -> cat
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


cat_or_dog_2.jpg -> cat
